In [ ]:
from diffusion_policy_gml.dataset import GmlDatasetNoSliding
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import pickle
from collections import defaultdict
import cv2
import tqdm.notebook as tqdm

In [ ]:
dataset_path_drawing = Path("data/gml_by_drawing_PRESERVE_ASPECT_CENTERED_003000.zarr")
dataset_path_stroke = Path("data/gml_by_stroke_PRESERVE_ASPECT_CENTERED_003000.zarr")

dataset = GmlDatasetNoSliding(dataset_path_drawing, -1, normalize=defaultdict(lambda: False), action_penlift=True, clip_embeddings=True)

In [ ]:
dataset[0]['obs']

In [ ]:
plt.plot(*dataset[0]['obs'].T)
plt.xlim(0, 1)
plt.ylim(0, 1)

In [ ]:
dataset[0]['action']

In [ ]:
dataset[0]['obs'][:34]

In [ ]:
len(dataset[0]['obs'])

In [ ]:
dataset[0]['obs'][:10]

In [ ]:
# Render into image
def render(obs, actions):
    image = np.ones((336, 336), dtype=np.uint8) * 255
    lifts = np.where(actions[:, 2])[0] + 1
    prev = 0
    for lift in lifts:
        for i in range(prev + 1, lift):
            x0, y0 = obs[i-1]
            x1, y1 = obs[i]
            x0, y0 = int(x0 * 336), int(y0 * 336)
            x1, y1 = int(x1 * 336), int(y1 * 336)
            image = cv2.line(image, (x0, y0), (x1, y1), 0, 3, cv2.LINE_AA)
        prev = lift
    return np.flipud(image)

im = render(dataset[0]['obs'], dataset[0]['action'])
k = 2
im = render(dataset[k]['obs'], dataset[k]['action'])
plt.imshow(im, cmap='gray')
cv2.imwrite('test.jpg', im);

In [ ]:
folder = Path('data/gml_renderings/')
folder.mkdir(exist_ok=True)
for drawing in tqdm.tqdm(dataset):
    im = render(drawing['obs'], drawing['action'])
    cv2.imwrite((folder / drawing["filename"]).as_posix(), im);